In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import regex as re
import emoji
from transformers import pipeline

e:\Naad\Projects\Real-Time Social Media Monitoring of Rupiah Exchange Rate Issues Using Transformer-Based NLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/processed/twitter_data_cleaned.csv")

In [3]:
df.head()

,id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length
0,2058239245148651767,kondisi negara udah bener bener diluar nalar\n...,0,0,0,0,2,2026-05-23,0,ensivo,230
1,2058239197652406671,ciecie joong jadi rupiah bgt didepan bininya 🤭...,0,0,0,0,2,2026-05-23,0,natachaintikk,71
2,2058239004710158506,membeli semua make up konser 0 rupiah karena c...,0,0,0,0,0,2026-05-23,0,3JSPACE,63
3,2058238886363668815,Nilai tukar rupiah gak disamain ma yen aja git...,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122
4,2058238081271308426,aduh aku bacanya ikutan kaya rupiah,0,0,0,0,3,2026-05-23,0,zss0ur,35


In [4]:
def preprocess(text):
    if pd.isna(text):

        return ""

    text=text.lower() # case folding
    text=re.sub(r"http\S+"," ",text) # url
    text=re.sub(r"@\w+"," ",text) # mention
    text=re.sub(r"#"," ",text) # hashtag
    text=re.sub(r"\brt\b"," ",text) # RT
    text=emoji.replace_emoji(text,'') # emoji
    text=re.sub(r"[^a-zA-Z0-9\s]"," ",text) # simbol
    text=re.sub(r"\s+"," ",text).strip() # spasi berlebih
    
    return text

df["clean_text"] = df['text'].apply(preprocess)

### Normalisasi

In [5]:
norm={
    " gk ":" tidak ",
    " ga ":" tidak ",
    " gak ":" tidak ",
    " yg ":" yang ",
    " dpt ":" dapat ",
    " bgt ":" banget ",
    " krn ":" karena ",
    " udh ":" sudah ",
    " tdk ":" tidak ",
    " gt ":" gitu ",
    " klo ":" kalau ",
    " kalo ":" kalau ",
    " aja ":" saja ",
    " nih ":" ini ",
    " ama ":" sama ",
    " ma ":" sama ",
    " siaapp ":" siap ",
    " bnr ":" benar ",
    " gpp ":" tidak apa-apa ",
    " bener ":" benar ",
    " dr ":" dari ",
    " hrs ":" harus ",
    " nnti ":" nanti ",
    " nnt ":" nanti ",
    " dgn ":" dengan ",
    " sm ":" sama ",
    " sy ":" saya ",
    " lg ":" lagi ",
    " hrsnya ":" seharusnya ",
    " hrsnha ":" seharusnya ",
    " hrsnyaa ":" seharusnya ",
    " hrsnyaaa ":" seharusnya ",
    " sblm ":" sebelum ",
    " sblmnya ":" sebelumnya ",
    " bgttt ":" banget ",
    " tpi ":" tapi ",
    " tp ":" tapi ",
    " bgtu ":" begitu ",
    " blm ":" belum ",
    " gmn ":" gimana ",
    " mls ":" malas ",
    "tp ":" tapi ",
    "tpi ":" tapi ",
    " kpd ":" kepada ",
    " brp ":" berapa ",
    " bs ":" bisa ",
    " mo ":" mau ",
    " krj ":" kerja "
}

def normalisasi(str_text):
  for i in norm:
    str_text = str_text.replace(i, norm[i])
  return str_text

df['clean_text'] = df['clean_text'].apply(lambda x: normalisasi(x))

In [6]:
df.head()

,id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length,clean_text
0,2058239245148651767,kondisi negara udah bener bener diluar nalar\n...,0,0,0,0,2,2026-05-23,0,ensivo,230,kondisi negara udah benar bener diluar nalar s...
1,2058239197652406671,ciecie joong jadi rupiah bgt didepan bininya 🤭...,0,0,0,0,2,2026-05-23,0,natachaintikk,71,ciecie joong jadi rupiah banget didepan bininya
2,2058239004710158506,membeli semua make up konser 0 rupiah karena c...,0,0,0,0,0,2026-05-23,0,3JSPACE,63,membeli semua make up konser 0 rupiah karena c...
3,2058238886363668815,Nilai tukar rupiah gak disamain ma yen aja git...,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122,nilai tukar rupiah tidak disamain sama yen saj...
4,2058238081271308426,aduh aku bacanya ikutan kaya rupiah,0,0,0,0,3,2026-05-23,0,zss0ur,35,aduh aku bacanya ikutan kaya rupiah


In [7]:
classifier = pipeline("sentiment-analysis",model="w11wo/indonesian-roberta-base-sentiment-classifier")

batch_size = 64
hasil_label=[]
hasil_score=[]

for i in tqdm(
    range(0, len(df), batch_size),
    desc="Proses Sentimen"
):

    teks_batch = (
        df["clean_text"]
        .iloc[i:i+batch_size]
        .fillna("")
        .astype(str)
        .tolist()
    )

    pred_batch = classifier(
        teks_batch,
        truncation=True,
        max_length=512
    )
    hasil_label.extend(
        [x["label"] for x in pred_batch]
    )

    hasil_score.extend(
        [x["score"] for x in pred_batch]
    )

    if len(hasil_label) % 1000 == 0:
        temp_df = df.iloc[:len(hasil_label)].copy()
        temp_df["sentimen_bert"] = hasil_label
        temp_df.to_csv(
            "../data/processed/checkpoint_sentiment.csv",
            index=False,
            )

df["sentimen_bert"]=hasil_label
df["confidence"]=hasil_score

Proses Sentimen: 100%|██████████| 138/138 [26:13<00:00, 11.40s/it]


In [8]:
df.head()

,id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length,clean_text,sentimen_bert,confidence
0,2058239245148651767,kondisi negara udah bener bener diluar nalar\n...,0,0,0,0,2,2026-05-23,0,ensivo,230,kondisi negara udah benar bener diluar nalar s...,negative,0.989148
1,2058239197652406671,ciecie joong jadi rupiah bgt didepan bininya 🤭...,0,0,0,0,2,2026-05-23,0,natachaintikk,71,ciecie joong jadi rupiah banget didepan bininya,positive,0.534576
2,2058239004710158506,membeli semua make up konser 0 rupiah karena c...,0,0,0,0,0,2026-05-23,0,3JSPACE,63,membeli semua make up konser 0 rupiah karena c...,neutral,0.952872
3,2058238886363668815,Nilai tukar rupiah gak disamain ma yen aja git...,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122,nilai tukar rupiah tidak disamain sama yen saj...,negative,0.777471
4,2058238081271308426,aduh aku bacanya ikutan kaya rupiah,0,0,0,0,3,2026-05-23,0,zss0ur,35,aduh aku bacanya ikutan kaya rupiah,negative,0.989753


In [9]:
df.to_csv("../data/processed/sentiment_data.csv",index=False)

In [10]:
df.sentimen_bert.value_counts()

sentimen_bert
negative    4080
neutral     3261
positive    1461
Name: count, dtype: int64